# Walmart Store Sales — XGBoost + SARIMA inference

Inference-only notebook using the registered `champion` raw-input pipeline.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "scikit-learn>=1.6,<2" "pandas>=2,<3" "wandb>=0.19,<1" "cloudpickle>=3,<4"

In [ ]:
from __future__ import annotations
import time
from pathlib import Path
import cloudpickle
import numpy as np
import pandas as pd
import wandb
from sklearn.pipeline import Pipeline

## Configuration

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/walmart_competition_data")
OUTPUT_DIR = Path("/content/drive/MyDrive/walmart_competition_inference/xgboost_sarima")
DOWNLOAD_DIR = Path("/content/artifacts/wandb_registry_xgboost_sarima")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
REGISTRY_ARTIFACT_URI = "wandb-registry-model/Walmart_XGBoost_SARIMA_Pipeline:champion"

## Load raw test data

In [ ]:
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
assert test_raw.columns.tolist() == ["Store", "Dept", "Date", "IsHoliday"]
assert not test_raw.duplicated(["Store", "Dept", "Date"]).any()
print({"rows": len(test_raw), "start": test_raw.Date.min(), "end": test_raw.Date.max()})

## Download champion pipeline from W&B Registry

In [ ]:
run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    name="xgboost-sarima-inference",
    job_type="inference",
    tags=["registry", "champion", "kaggle-submission"],
)
artifact = run.use_artifact(REGISTRY_ARTIFACT_URI, type="model")
artifact_dir = Path(artifact.download(root=DOWNLOAD_DIR))
pipeline_files = list(artifact_dir.rglob("*.pkl"))
assert len(pipeline_files) == 1
with pipeline_files[0].open("rb") as file:
    pipeline = cloudpickle.load(file)
assert isinstance(pipeline, Pipeline)
assert list(pipeline.named_steps) == ["feature_engineering", "hybrid_model"]

## Predict directly from raw test rows

In [ ]:
started = time.perf_counter()
predictions = np.asarray(pipeline.predict(test_raw), dtype=float)
elapsed = time.perf_counter() - started
assert predictions.shape == (len(test_raw),) and np.isfinite(predictions).all()
summary = {
    "rows": len(predictions),
    "seconds": elapsed,
    "mean": float(predictions.mean()),
    "std": float(predictions.std()),
    "min": float(predictions.min()),
    "max": float(predictions.max()),
}
run.log({f"inference/{k}": v for k, v in summary.items()})
display(pd.Series(summary, name="value").to_frame())

## Create and log Kaggle submission

In [ ]:
submission = pd.DataFrame(
    {
        "Id": test_raw.Store.astype(str)
        + "_"
        + test_raw.Dept.astype(str)
        + "_"
        + test_raw.Date.dt.strftime("%Y-%m-%d"),
        "Weekly_Sales": predictions,
    }
)
assert submission.Id.is_unique and submission.Weekly_Sales.notna().all()
submission_path = OUTPUT_DIR / "submission_xgboost_sarima_champion.csv"
submission.to_csv(submission_path, index=False)
submission_artifact = wandb.Artifact(
    "xgboost-sarima-registry-submission",
    type="submission",
    metadata={**summary, "registry_artifact": REGISTRY_ARTIFACT_URI},
)
submission_artifact.add_file(str(submission_path))
logged = run.log_artifact(submission_artifact)
logged.wait()
run.finish()
print(submission_path)
display(submission.head())